# EDA — Cloud Spot Instance Price Optimizer

Exploratory analysis of AWS GPU spot price data.
Reads from `data/processed/base_dataset.parquet` (output of the data pipeline).

Sections:
1. Data overview
2. Time dimension
3. Price distributions
4. Price volatility
5. Price over time for a specific instance
6. Volatility across availability zones
7. Day of week analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 1. Load Data

In [ ]:
df = pd.read_parquet(Path('data/processed/base_dataset.parquet'))
print(df.shape)
df.head()

In [ ]:
# Missing values and unique counts
print(df.isnull().sum())
print()
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

## 2. Time Dimension

In [ ]:
# Date range
print(f"Start: {df['timestamp'].min()}")
print(f"End:   {df['timestamp'].max()}")

# How frequently do prices change per instance/zone?
df_sorted = df.sort_values(['instance_type', 'availability_zone', 'timestamp'])
df_sorted['time_since_last_change'] = df_sorted.groupby(
    ['instance_type', 'availability_zone']
)['timestamp'].diff()

print()
print(df_sorted['time_since_last_change'].describe())

## 3. Price Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['price'].hist(bins=50, ax=axes[0])
axes[0].set_title('Price distribution (all instances)')
axes[0].set_xlabel('Price ($/hr)')

df['price'].hist(bins=50, log=True, ax=axes[1])
axes[1].set_title('Price distribution (log scale)')
axes[1].set_xlabel('Price ($/hr)')

plt.tight_layout()
plt.show()

# By instance family
df['instance_family'] = df['instance_type'].str.split('.').str[0]
df.boxplot(column='price', by='instance_family', figsize=(12, 5))
plt.title('Price distribution by instance family')
plt.suptitle('')
plt.ylabel('Price ($/hr)')
plt.show()

## 4. Price Volatility by Instance Type

In [ ]:
volatility = df.groupby('instance_type')['price'].agg([
    'mean', 'std', 'min', 'max',
    lambda x: (x.max() - x.min()) / x.mean() * 100
]).round(4)

volatility.columns = ['mean_price', 'std', 'min_price', 'max_price', 'pct_range']
volatility = volatility.sort_values('pct_range', ascending=False)
print(volatility.head(20))

## 5. Price Over Time — Single Instance

In [ ]:
instance = 'g5.2xlarge'
zone = 'use1-az1'

subset = df[
    (df['instance_type'] == instance) &
    (df['availability_zone'] == zone)
].sort_values('timestamp')

plt.figure(figsize=(14, 4))
plt.plot(subset['timestamp'], subset['price'], marker='o', markersize=2)
plt.title(f'Spot price over time: {instance} in {zone}')
plt.xlabel('Date')
plt.ylabel('Price ($/hr)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Number of price changes: {len(subset)}")
print(f"Price range: ${subset['price'].min():.4f} - ${subset['price'].max():.4f}")
print(f"Mean price: ${subset['price'].mean():.4f}")

## 6. Volatility Across Availability Zones

In [ ]:
instance = 'g5.2xlarge'
subset = df[df['instance_type'] == instance]

zone_stats = subset.groupby('availability_zone')['price'].agg(['mean', 'std', 'min', 'max'])
zone_stats['volatility'] = zone_stats['std'] / zone_stats['mean']
print(zone_stats.sort_values('volatility', ascending=False))

## 7. Day of Week Analysis

Hypothesis: ML training jobs cluster around business hours on weekdays, increasing demand and pushing spot prices up. Prices should be lower on weekends when demand drops.

In [ ]:
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

subset = df[df['instance_type'] == 'g5.2xlarge'].copy()
subset['day_of_week'] = subset['timestamp'].dt.dayofweek

daily_avg = subset.groupby('day_of_week')['price'].agg(['mean', 'std']).reset_index()
daily_avg['day_name'] = daily_avg['day_of_week'].map(dict(enumerate(day_names)))

plt.figure(figsize=(10, 4))
plt.bar(daily_avg['day_name'], daily_avg['mean'], yerr=daily_avg['std'], capsize=5)
plt.title('Average spot price by day of week: g5.2xlarge')
plt.ylabel('Price ($/hr)')
plt.xlabel('Day of week')
plt.show()

### Kruskal-Wallis Test

Tests whether price differences across days of the week are statistically significant.

In [ ]:
from scipy import stats

groups = [subset[subset['day_of_week'] == day]['price'].values
          for day in range(7)]

stat, p_value = stats.kruskal(*groups)
print(f"Kruskal-Wallis statistic: {stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: statistically significant difference across days")
else:
    print("Result: no statistically significant difference across days")

### Dunn Post-Hoc Test

If Kruskal-Wallis is significant, Dunn's test identifies which specific day pairs differ.

In [ ]:
%pip install scikit_posthocs -q

In [ ]:
from scikit_posthocs import posthoc_dunn

prices_with_days = subset[['day_of_week', 'price']]
dunn_results = posthoc_dunn(prices_with_days, val_col='price', group_col='day_of_week')
print(dunn_results)

### Granger Causality Test

Tests whether day of week Granger-causes price changes. Look for F-test p-values consistently below 0.05 across multiple lag values.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Resample to hourly for a consistent time index
ts = subset.set_index('timestamp')['price'].resample('1h').mean().ffill()
ts_df = ts.reset_index()
ts_df['day_of_week'] = ts_df['timestamp'].dt.dayofweek

# First-difference price to remove trend (required for stationarity)
ts_df['price_diff'] = ts_df['price'].diff()
data = ts_df[['price_diff', 'day_of_week']].dropna()

results = grangercausalitytests(data, maxlag=7, verbose=True)

### Deductive Reasoning

The causal mechanism behind day-of-week price patterns:

- ML training jobs are disproportionately launched by engineering teams during business hours on weekdays
- This creates higher demand for GPU instances Monday–Friday
- Higher demand reduces AWS spare capacity, pushing spot prices up
- On weekends, demand drops, capacity frees up, and prices fall

This is a testable, domain-grounded hypothesis that the statistical tests above either support or refute.